In [2]:
!pip install rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 18.1 MB/s eta 0:00:00


**Chargement en CSV de Teny Malagasy**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import requests
import re
import json
from rapidfuzz import process, fuzz

CHEMIN_CSV = "/content/drive/MyDrive/Colab Notebooks/malagasy_dictionary.csv"

# ── Chargement ─────────────────────────────────────────────────────────
try:
    df = pd.read_csv(CHEMIN_CSV, encoding="utf-8")
except UnicodeDecodeError:
    df = pd.read_csv(CHEMIN_CSV, encoding="latin-1")

# ── Correction encodage ────────────────────────────────────────────────
def fix_encoding(texte):
    if not isinstance(texte, str):
        return texte
    remplacements = {
        "Ã±": "ñ", "Ã ": "à", "Ã©": "é",
        "Ã¨": "è", "Ã´": "ô", "Ã»": "û",
        "â€™": "'", "â€": "–",
    }
    for mal, bon in remplacements.items():
        texte = texte.replace(mal, bon)
    return texte

df["racine"] = df["racine"].apply(fix_encoding)
df["mot"]    = df["mot"].apply(fix_encoding)
df = df.dropna().drop_duplicates()
df = df[df["mot"].str.len() >= 2]

# ── Structures issues du CSV ───────────────────────────────────────────
MOTS_TENY       = set(df["mot"].str.lower().str.strip())
RACINES_TENY    = set(df["racine"].str.lower().str.strip())
MOT_VERS_RACINE = dict(zip(
    df["mot"].str.lower().str.strip(),
    df["racine"].str.lower().str.strip()
))

print(f"[tenymalagasy] {len(MOTS_TENY)} mots | {len(RACINES_TENY)} racines")
print(f"               Mapping mot→racine : {len(MOT_VERS_RACINE)} entrées")

Mounted at /content/drive
[tenymalagasy] 43700 mots | 6435 racines
               Mapping mot→racine : 43700 entrées


**Scrapping de Wikipedia MG**

In [4]:
def fetch_wikipedia_mg(nb_articles=200):
    mots, phrases = set(), []
    url     = "https://mg.wikipedia.org/w/api.php"
    headers = {"User-Agent": "NLPMalagasyBot/1.0 (projet etudiant)"}

    try:
        r = requests.get(url, headers=headers, params={
            "action": "query", "list": "random",
            "rnnamespace": 0, "rnlimit": nb_articles, "format": "json"
        }, timeout=15)
        if r.status_code != 200:
            print(f"[Wikipedia] HTTP {r.status_code}")
            return mots, phrases
        pages = r.json()["query"]["random"]
    except Exception as e:
        print(f"[Wikipedia] Erreur : {e}")
        return mots, phrases

    ok, echec = 0, 0
    for page in pages:
        try:
            rd = requests.get(url, headers=headers, params={
                "action": "query", "pageids": page["id"],
                "prop": "extracts", "explaintext": True, "format": "json"
            }, timeout=10)
            if rd.status_code != 200 or not rd.text.strip():
                echec += 1
                continue
            texte = rd.json()["query"]["pages"][str(page["id"])].get("extract", "")
            if not texte.strip():
                echec += 1
                continue
            for phrase in re.split(r'[.!?\n]+', texte):
                phrase = phrase.strip()
                if 10 < len(phrase) < 300:
                    phrases.append(phrase)
            for mot in re.findall(r"[a-zA-Zñ']{2,}", texte.lower()):
                mots.add(mot)
            ok += 1
        except Exception:
            echec += 1
    print(f"[Wikipedia MG] {ok} pages OK | {len(mots)} mots | {len(phrases)} phrases")
    return mots, phrases

mots_wiki, phrases_wiki = fetch_wikipedia_mg(nb_articles=200)

[Wikipedia MG] 131 pages OK | 2130 mots | 1448 phrases


**Scrapping de Bible Malagasy**

In [5]:
def fetch_bible_malagasy():
    mots, phrases = set(), []
    url = "https://raw.githubusercontent.com/christos-c/bible-corpus/master/bibles/Malagasy.xml"
    try:
        r = requests.get(url, timeout=30)
        if r.status_code != 200 or not r.text.strip():
            print(f"[Bible] HTTP {r.status_code}")
            return mots, phrases
        texte_brut = re.sub(r'<[^>]+>', ' ', r.text)
        texte_brut = re.sub(r'\s+', ' ', texte_brut)
        for ligne in texte_brut.split('.'):
            ligne = ligne.strip()
            if len(ligne) > 10:
                phrases.append(ligne)
                for mot in re.findall(r"[a-zA-Zñ']{2,}", ligne.lower()):
                    mots.add(mot)
        print(f"[Bible MG] {len(mots)} mots | {len(phrases)} phrases")
    except Exception as e:
        print(f"[Bible] Erreur : {e}")
    return mots, phrases

mots_bible, phrases_bible = fetch_bible_malagasy()

[Bible MG] 26603 mots | 27666 phrases


**Fusion et nettoyage**

In [8]:
def est_mot_valide(mot: str) -> bool:
    VOYELLES = set("aeiou")
    if not (2 <= len(mot) <= 25):
        return False
    if not any(c in VOYELLES for c in mot):
        return False
    if not re.match(r"^[a-zA-Zñ']+$", mot):
        return False
    return True

# ── Dictionnaire unifié ────────────────────────────────────────────────
DICTIONNAIRE = set()

# Source 1 : tenymalagasy (mots + racines)
for mot in MOTS_TENY | RACINES_TENY:
    if est_mot_valide(mot):
        DICTIONNAIRE.add(mot.lower().strip())

# Source 2 : Wikipedia
for mot in mots_wiki:
    mot_propre = mot.lower().strip().strip("'")
    if est_mot_valide(mot_propre):
        DICTIONNAIRE.add(mot_propre)

# Source 3 : Bible
for mot in mots_bible:
    mot_propre = mot.lower().strip().strip("'")
    if est_mot_valide(mot_propre):
        DICTIONNAIRE.add(mot_propre)

# ── Corpus de phrases unifié ───────────────────────────────────────────
CORPUS_PHRASES = list(dict.fromkeys(
    p.strip() for p in phrases_wiki + phrases_bible
    if len(p.strip().split()) >= 3
))

# ── Pour rapidfuzz ─────────────────────────────────────────────────────
DICO_LIST = list(DICTIONNAIRE)

print(f"\n DICTIONNAIRE final : {len(DICTIONNAIRE)} mots")
print(f"   dont racines tenymalagasy : {len(RACINES_TENY)}")
print(f"   dont mots Wikipedia       : {len(mots_wiki)}")
print(f"   dont mots Bible           : {len(mots_bible)}")
print(f" CORPUS PHRASES     : {len(CORPUS_PHRASES)} phrases")


 DICTIONNAIRE final : 72075 mots
   dont racines tenymalagasy : 6435
   dont mots Wikipedia       : 2130
   dont mots Bible           : 26603
 CORPUS PHRASES     : 27347 phrases


**Stopwords, tokenisation**

In [9]:
STOPWORDS_MG = {
    "ny", "sy", "no", "na", "fa", "ka", "dia", "ary",
    "ao", "an", "am", "in", "ho", "de", "aza", "ve",
    "re", "tsy", "amin", "avy", "any", "eny", "izay",
    "izy", "aho", "ianao", "isika", "ilay", "ireo",
    "ity", "io", "iny", "koa", "ihany", "foana", "nefa",
    "saingy", "kanefa", "raha", "satria", "mba", "aza",
}

def tokeniser(texte: str) -> list:
    """
    Tokeniseur malagasy qui gère :
    - Le trait d'union '-' (fampianaran-teny → 2 tokens)
    - L'apostrophe "'" (amin'ny → 2 tokens)
    - La ponctuation en fin/début de token
    """
    # Normalise les apostrophes typographiques
    texte = texte.replace("\u2019", "'").replace("\u2018", "'")

    tokens = []
    # Split initial sur espaces et ponctuation forte
    for bloc in re.split(r'[\s,;:!?."()\[\]]+', texte):
        bloc = bloc.strip()
        if not bloc:
            continue
        # Split sur trait d'union
        for partie in bloc.split("-"):
            # Split sur apostrophe
            for sp in partie.split("'"):
                sp = sp.lower().strip().strip("'")
                if len(sp) >= 2:
                    tokens.append(sp)

    return tokens

**Module 1: Correcteur orthographique**

In [11]:
DICO_LIST = list(DICTIONNAIRE)  # pour rapidfuzz

def est_correct(mot: str) -> bool:
    return mot.lower().strip() in DICTIONNAIRE

def suggerer(mot: str, nb=3, seuil=70) -> list:
    resultats = process.extract(
        mot.lower().strip(),
        DICO_LIST,
        scorer=fuzz.ratio,
        limit=nb
    )
    return [(m, s) for m, s, _ in resultats if s >= seuil]

def corriger_texte(texte: str) -> dict:
    """Retourne {mot_incorrect: [(suggestion, score), ...]}"""
    erreurs = {}
    for mot in set(tokeniser(texte)):
        if mot in STOPWORDS_MG:
            continue
        if not est_correct(mot):
            erreurs[mot] = suggerer(mot)
    return erreurs


# ── Test ───────────────────────────────────────────────────────────────
print("=== Module 1 : Correcteur ===\n")
tests = [
    "Ny fitiavana sy ny fampianaran-teny no zava-dehibe",
    "Amin'ny alina dia matory ny ankizy malagasi",
]
for t in tests:
    print(f" {t}")
    erreurs = corriger_texte(t)
    if not erreurs:
        print("   Aucune erreur")
    for mot, sugg in erreurs.items():
        print(f"   Erreur '{mot}' → {[s for s,_ in sugg] or ['—']}")
    print()

=== Module 1 : Correcteur ===

 Ny fitiavana sy ny fampianaran-teny no zava-dehibe
   Erreur 'fampianaran' → ['fampianarany', 'fampianarana', 'fampianaranao']

 Amin'ny alina dia matory ny ankizy malagasi
   Erreur 'malagasi' → ['malagasy', 'malanga', 'malai']



**Module 2: Verification phonotactique/REGEX**

In [12]:
from dataclasses import dataclass
from typing import List

@dataclass
class ErreurPhono:
    mot: str
    regle: str
    description: str

REGLES_PHONOTACTIQUES = [
    {
        "id": "COMB_INTERDIT",
        "pattern": re.compile(r'nb|mk|dt|bp|sz', re.I),
        "description": "Combinaison de consonnes inexistante en malagasy"
    },
    {
        "id": "NK_DEBUT",
        "pattern": re.compile(r'^nk|nb|y|ml|nt|np', re.I),
        "description": "Début de mot  interdit"
    },
    {
        "id": "LETTRE_ETRANGERE",
        "pattern": re.compile(r'[cquwx]', re.I),
        "description": "Lettre absente de l'alphabet malagasy (c, q, u, w, x)"
    },
    {
        "id": "DOUBLE_CONSONNE",
        "pattern": re.compile(r'([bcdfgjklmnprstvz])\1', re.I),
        "description": "Double consonne inhabituelle"
    },
    {
        "id": "FIN_CONSONNE",
        "pattern": re.compile(r'[bcdfgjklmprstv]$', re.I),
        "description": "Fin de mot atypique (malagasy finit par voyelle ou -ny)"
    },
]

def verifier_phonotactique(mot: str) -> List[ErreurPhono]:
    # Si le mot est dans le dictionnaire, il est correct par définition
    if mot.lower() in DICTIONNAIRE:
        return []
    erreurs = []
    for regle in REGLES_PHONOTACTIQUES:
        if regle["pattern"].search(mot.strip()):
            erreurs.append(ErreurPhono(mot, regle["id"], regle["description"]))
    return erreurs


# ── Test ───────────────────────────────────────────────────────────────
print("=== Module 2 : Phonotactique ===\n")
tests_phono = ["manosika", "ankizy", "nbato", "foxer", "mkcola", "trano"]
for mot in tests_phono:
    e = verifier_phonotactique(mot)
    if e:
        print(f"  Incorrect: '{mot}' → [{e[0].regle}] {e[0].description}")
    else:
        print(f"  Correct '{mot}'")

=== Module 2 : Phonotactique ===

  Correct 'manosika'
  Correct 'ankizy'
  Incorrect: 'nbato' → [COMB_INTERDIT] Combinaison de consonnes inexistante en malagasy
  Incorrect: 'foxer' → [LETTRE_ETRANGERE] Lettre absente de l'alphabet malagasy (c, q, u, w, x)
  Incorrect: 'mkcola' → [COMB_INTERDIT] Combinaison de consonnes inexistante en malagasy
  Correct 'trano'


**Module 3: Lemmatisation**

In [13]:
PREFIXES = [
    "mpampi", "mpanao", "mpan", "mampian", "mampi", "mamp",
    "mifan", "maha", "man", "mam", "mi", "ma",
    "fampi", "famp", "fan", "fam", "fi", "fa",
    "tamin", "tan", "am", "an", "ifan", "i",
]
SUFFIXES = ["andriny", "indriny", "iana", "iny", "ana", "ina", "na", "ny"]

MUTATIONS = {
    "d":  ["r", "l", "dr"],
    "j":  ["z", "j"],
    "nd": ["l", "r", "nd"],
    "dr": ["r"],
    "b":  ["v", "b"],
    "mp": ["p", "mp"],
}

def appliquer_mutations(mot: str) -> list:
    candidats = []
    for mutation, origines in MUTATIONS.items():
        if mot.startswith(mutation):
            reste = mot[len(mutation):]
            for origine in origines:
                candidat = origine + reste
                if candidat in DICTIONNAIRE:
                    candidats.append(candidat)
    return candidats

def restaurer_voyelle_finale(mot: str) -> str:
    if mot and mot[-1] not in "aeiou":
        base = mot[:-1] if mot.endswith("n") else mot
        for v in ["a", "e", "i", "o"]:
            if base + v in DICTIONNAIRE:
                return base + v
        for v in ["a", "e", "i", "o"]:
            if mot + v in DICTIONNAIRE:
                return mot + v
    return mot

def suffixe_na_est_valide(mot: str, racine_candidate: str) -> bool:
    return racine_candidate in DICTIONNAIRE


def lemmatiser(mot: str) -> dict:
    mot_c = mot.lower().strip()

    # ── N0 : lookup direct dans MOT_VERS_RACINE (tenymalagasy) ────────
    # C'est le niveau le plus fiable — racine garantie par le dictionnaire
    if mot_c in MOT_VERS_RACINE:
        return {
            "racine":    MOT_VERS_RACINE[mot_c],
            "methode":   "lookup_direct",
            "confiance": "haute"
        }

    # ── N0bis : le mot est lui-même une racine dans tenymalagasy ───────
    if mot_c in RACINES_TENY:
        return {
            "racine":    mot_c,
            "methode":   "racine_directe",
            "confiance": "haute"
        }

    # ── Restauration voyelle finale avant analyse ──────────────────────
    restaure = restaurer_voyelle_finale(mot_c)

    # Vérifie aussi la forme restaurée dans MOT_VERS_RACINE
    if restaure != mot_c and restaure in MOT_VERS_RACINE:
        return {
            "racine":    MOT_VERS_RACINE[restaure],
            "methode":   "lookup_restaure",
            "confiance": "haute"
        }

    formes = [mot_c, restaure] if restaure != mot_c else [mot_c]
    candidats = []

    for forme in formes:

        # ── N1 : règles morphologiques ─────────────────────────────────
        for pfx in sorted(PREFIXES, key=len, reverse=True):
            if forme.startswith(pfx) and len(forme) > len(pfx) + 2:
                reste = forme[len(pfx):]

                for sfx in sorted(SUFFIXES, key=len, reverse=True):
                    if reste.endswith(sfx) and len(reste) > len(sfx) + 1:
                        racine = reste[:-len(sfx)]
                        if sfx == "na" and not suffixe_na_est_valide(forme, racine):
                            continue
                        if racine in DICTIONNAIRE:
                            candidats.append((racine, len(pfx) + len(sfx), "regles"))

                if reste in DICTIONNAIRE:
                    candidats.append((reste, len(pfx), "regles"))

                for racine_mutee in appliquer_mutations(reste):
                    candidats.append((racine_mutee, len(pfx) + 1, "mutation+prefixe"))

                for sfx in sorted(SUFFIXES, key=len, reverse=True):
                    if reste.endswith(sfx) and len(reste) > len(sfx) + 1:
                        reste_sans_sfx = reste[:-len(sfx)]
                        if sfx == "na" and not suffixe_na_est_valide(forme, reste_sans_sfx):
                            continue
                        for racine_mutee in appliquer_mutations(reste_sans_sfx):
                            candidats.append((racine_mutee, len(pfx) + len(sfx) + 1, "mutation+prefixe+suffixe"))

        for sfx in sorted(SUFFIXES, key=len, reverse=True):
            if forme.endswith(sfx) and len(forme) > len(sfx) + 2:
                racine = forme[:-len(sfx)]
                if sfx == "na" and not suffixe_na_est_valide(forme, racine):
                    continue
                if racine in DICTIONNAIRE:
                    candidats.append((racine, len(sfx), "regles"))

        # ── N2 : mutations directes ────────────────────────────────────
        for racine_mutee in appliquer_mutations(forme):
            candidats.append((racine_mutee, 1, "mutation_directe"))

    if candidats:
        meilleur = min(candidats, key=lambda x: len(x[0]))
        return {
            "racine":    meilleur[0],
            "methode":   meilleur[2],
            "confiance": "haute" if meilleur[0] in DICTIONNAIRE else "moyenne"
        }

    # ── N3 : forme restaurée seule ────────────────────────────────────
    if restaure != mot_c and restaure in DICTIONNAIRE:
        return {"racine": restaure, "methode": "restauration", "confiance": "haute"}

    # ── N4 : mot dans le dico sans décomposition ──────────────────────
    if mot_c in DICTIONNAIRE:
        return {"racine": mot_c, "methode": "direct", "confiance": "haute"}

    # ── N5 : rien trouvé ──────────────────────────────────────────────
    return {"racine": mot_c, "methode": "non_trouvé", "confiance": "faible"}


# ── Test ───────────────────────────────────────────────────────────────
tests = [
    ("alina",     "alina"),
    ("fomban",    "fomba"),
    ("toetr",     "toetra"),
    ("drazana",   "razana"),
    ("danja",     "lanja"),
    ("fitiavana", "tia"),
    ("mianatra",  "anatra"),
    ("malagasy",  "malagasy"),
]

print(f"{'Mot':<18} {'Lemme trouvé':<18} {'Attendu':<15} {'Méthode':<25} {'OK?'}")
print("-" * 80)
for mot, attendu in tests:
    r = lemmatiser(mot)
    ok = "" if r["racine"] == attendu else " "
    print(f"{mot:<18} {r['racine']:<18} {attendu:<15} {r.get('methode','?'):<25} {ok}")

Mot                Lemme trouvé       Attendu         Méthode                   OK?
--------------------------------------------------------------------------------
alina              alina              alina           racine_directe            
fomban             omba               fomba           lookup_restaure            
toetr              toetra             toetra          restauration              
drazana            razana             razana          mutation_directe          
danja              lanja              lanja           mutation_directe          
fitiavana          tia                tia             lookup_direct             
mianatra           anatra             anatra          lookup_direct             
malagasy           malagasy           malagasy        direct                    


**Exemple concret**

In [15]:
def analyser_phrase(texte: str) -> None:
    """
    Analyse complète d'une phrase malagasy :
    - Tokenisation (gère - et ')
    - M1 : correction orthographique
    - M2 : vérification phonotactique
    - M3 : lemmatisation
    """
    print("=" * 65)
    print(f"TEXTE : {texte}")
    print("=" * 65)

    tokens = tokeniser(texte)
    print(f"Tokens : {tokens}\n")

    vus = set()
    for mot in tokens:
        if mot in vus:
            continue
        vus.add(mot)

        # Stopword
        if mot in STOPWORDS_MG:
            print(f"    '{mot}' — mot grammatical")
            continue

        print(f"\n   '{mot}'")

        # ── M1 : Orthographe ──────────────────────────────────────────
        restaure = restaurer_voyelle_finale(mot.lower())
        if est_correct(mot):
            if restaure != mot.lower() and restaure in DICTIONNAIRE:
                print(f"     Correct (forme complète : '{restaure}')")
            else:
                print(f"      Correct")
        else:
            sugg = suggerer(mot)
            print(f"      Inconnu | Suggestions : {[s for s, _ in sugg] or ['—']}")

        # ── M2 : Phonotactique ────────────────────────────────────────
        erreurs_phono = verifier_phonotactique(mot)
        for e in erreurs_phono:
            print(f"       [{e.regle}] {e.description}")

        # ── M3 : Lemmatisation ────────────────────────────────────────
        r = lemmatiser(mot)
        if r["methode"] != "non_trouvé":
            # N'affiche le lemme que s'il est différent du mot original
            if r["racine"] != mot.lower() and r["racine"] != restaure.lower():
                print(f"      Lemme : '{r['racine']}' "
                      f"({r['methode']}, confiance {r['confiance']})")
            else:
                print(f"      Lemme : '{r['racine']}' (racine de base)")
        else:
            print(f"      Lemme inconnu")

    print()


# ── Tests avec des phrases variées ────────────────────────────────────
phrases_test = [
    # Phrase simple
    "Ny ankizy mianatra ao amin'ny sekoly",

    # Mots composés avec trait d'union
    "Ny fomban-drazana malagasy dia manan-danja",

    # Apostrophe de liaison
    "Ny toetr'andro androany dia tsara",

    # Mots dérivés complexes
    "Ny fitiavana sy ny fampianaran-teny no zava-dehibe",

    # Phrase avec erreur intentionnelle
    "Ny ankizy malagasi dia mianatra foxer",
]

for phrase in phrases_test:
    analyser_phrase(phrase)

TEXTE : Ny ankizy mianatra ao amin'ny sekoly
Tokens : ['ny', 'ankizy', 'mianatra', 'ao', 'amin', 'ny', 'sekoly']

    'ny' — mot grammatical

   'ankizy'
      Correct
      Lemme : 'ankizy' (racine de base)

   'mianatra'
      Correct
      Lemme : 'anatra' (lookup_direct, confiance haute)
    'ao' — mot grammatical
    'amin' — mot grammatical

   'sekoly'
      Correct
      Lemme : 'sekoly' (racine de base)

TEXTE : Ny fomban-drazana malagasy dia manan-danja
Tokens : ['ny', 'fomban', 'drazana', 'malagasy', 'dia', 'manan', 'danja']

    'ny' — mot grammatical

   'fomban'
     Correct (forme complète : 'fombana')
      Lemme : 'omba' (lookup_restaure, confiance haute)

   'drazana'
      Correct
      Lemme : 'razana' (mutation_directe, confiance haute)

   'malagasy'
      Correct
      Lemme : 'malagasy' (racine de base)
    'dia' — mot grammatical

   'manan'
     Correct (forme complète : 'manao')
      Lemme : 'tao' (lookup_restaure, confiance haute)

   'danja'
      Correct


**Import en pickle**

In [16]:
import pickle

# ── Tout ce qui constitue le "modèle" de lemmatisation ────────────────
modele = {
    "DICTIONNAIRE":    DICTIONNAIRE,
    "RACINES_TENY":    RACINES_TENY,
    "MOT_VERS_RACINE": MOT_VERS_RACINE,
    "DICO_LIST":       DICO_LIST,
    "CORPUS_PHRASES":  CORPUS_PHRASES,
    "PREFIXES":        PREFIXES,
    "SUFFIXES":        SUFFIXES,
    "MUTATIONS":       MUTATIONS,
    "STOPWORDS_MG":    STOPWORDS_MG,
}

# ── Sauvegarde dans Drive ──────────────────────────────────────────────
CHEMIN_PKL = "/content/drive/MyDrive/modele_nlp_malagasy.pkl"

with open(CHEMIN_PKL, "wb") as f:
    pickle.dump(modele, f)

print(f"✅ Modèle sauvegardé : {CHEMIN_PKL}")
print(f"\nContenu sauvegardé :")
for cle, valeur in modele.items():
    if isinstance(valeur, (set, list, dict)):
        print(f"  {cle:<20} → {len(valeur)} éléments")
    else:
        print(f"  {cle:<20} → {valeur}")

✅ Modèle sauvegardé : /content/drive/MyDrive/modele_nlp_malagasy.pkl

Contenu sauvegardé :
  DICTIONNAIRE         → 72075 éléments
  RACINES_TENY         → 6435 éléments
  MOT_VERS_RACINE      → 43700 éléments
  DICO_LIST            → 72075 éléments
  CORPUS_PHRASES       → 27347 éléments
  PREFIXES             → 24 éléments
  SUFFIXES             → 8 éléments
  MUTATIONS            → 6 éléments
  STOPWORDS_MG         → 41 éléments


**Utilisation du modèle pickle**

In [18]:
import pickle
from rapidfuzz import process, fuzz
import re
from dataclasses import dataclass
from typing import List

# ── Chargement du modèle ───────────────────────────────────────────────
CHEMIN_PKL = "/content/drive/MyDrive/modele_nlp_malagasy.pkl"

with open(CHEMIN_PKL, "rb") as f:
    modele = pickle.load(f)

# ── Restauration des variables ─────────────────────────────────────────
DICTIONNAIRE    = modele["DICTIONNAIRE"]
RACINES_TENY    = modele["RACINES_TENY"]
MOT_VERS_RACINE = modele["MOT_VERS_RACINE"]
DICO_LIST       = modele["DICO_LIST"]
CORPUS_PHRASES  = modele["CORPUS_PHRASES"]
PREFIXES        = modele["PREFIXES"]
SUFFIXES        = modele["SUFFIXES"]
MUTATIONS       = modele["MUTATIONS"]
STOPWORDS_MG    = modele["STOPWORDS_MG"]

print(f" Modèle chargé")
print(f"   Dictionnaire : {len(DICTIONNAIRE)} mots")
print(f"   Racines      : {len(RACINES_TENY)} racines")
print(f"   Mapping      : {len(MOT_VERS_RACINE)} entrées mot→racine")
print(f"   Corpus       : {len(CORPUS_PHRASES)} phrases")

 Modèle chargé
   Dictionnaire : 72075 mots
   Racines      : 6435 racines
   Mapping      : 43700 entrées mot→racine
   Corpus       : 27347 phrases


In [19]:
# ── Toutes les fonctions utilitaires ──────────────────────────────────

# ── Tokeniseur ────────────────────────────────────────────────────────
def tokeniser(texte: str) -> list:
    texte = texte.replace("\u2019", "'").replace("\u2018", "'")
    tokens = []
    for bloc in re.split(r'[\s,;:!?."()\[\]]+', texte):
        bloc = bloc.strip()
        if not bloc:
            continue
        for partie in bloc.split("-"):
            for sp in partie.split("'"):
                sp = sp.lower().strip().strip("'")
                if len(sp) >= 2:
                    tokens.append(sp)
    return tokens


# ── Module 1 : Correction orthographique ──────────────────────────────
def restaurer_voyelle_finale(mot: str) -> str:
    if mot and mot[-1] not in "aeiou":
        base = mot[:-1] if mot.endswith("n") else mot
        for v in ["a", "e", "i", "o"]:
            if base + v in DICTIONNAIRE:
                return base + v
        for v in ["a", "e", "i", "o"]:
            if mot + v in DICTIONNAIRE:
                return mot + v
    return mot

def est_correct(mot: str) -> bool:
    mot_c = mot.lower().strip()
    if mot_c in DICTIONNAIRE:
        return True
    restaure = restaurer_voyelle_finale(mot_c)
    if restaure != mot_c and restaure in DICTIONNAIRE:
        return True
    return False

def suggerer(mot: str, nb=3, seuil=70) -> list:
    resultats = process.extract(
        mot.lower().strip(), DICO_LIST,
        scorer=fuzz.ratio, limit=nb
    )
    return [(m, s) for m, s, _ in resultats if s >= seuil]

def corriger_texte(texte: str) -> dict:
    erreurs = {}
    for mot in set(tokeniser(texte)):
        if mot in STOPWORDS_MG:
            continue
        if not est_correct(mot):
            erreurs[mot] = suggerer(mot)
    return erreurs


# ── Module 2 : Vérification phonotactique ─────────────────────────────
@dataclass
class ErreurPhono:
    mot:         str
    regle:       str
    description: str

REGLES_PHONOTACTIQUES = [
    {
        "id":          "COMB_INTERDIT",
        "pattern":     re.compile(r'nb|mk|dt|bp|sz', re.I),
        "description": "Combinaison de consonnes inexistante en malagasy"
    },
    {
        "id":          "NK_DEBUT",
        "pattern":     re.compile(r'^nk', re.I),
        "description": "Début de mot par 'nk' interdit"
    },
    {
        "id":          "LETTRE_ETRANGERE",
        "pattern":     re.compile(r'[cquwx]', re.I),
        "description": "Lettre absente de l'alphabet malagasy (c, q, u, w, x)"
    },
    {
        "id":          "DOUBLE_CONSONNE",
        "pattern":     re.compile(r'([bcdfgjklmnprstvz])\1', re.I),
        "description": "Double consonne inhabituelle"
    },
    {
        "id":          "FIN_CONSONNE",
        "pattern":     re.compile(r'[bcdfgjklmprstv]$', re.I),
        "description": "Fin de mot atypique (malagasy finit par voyelle ou -ny)"
    },
]

def verifier_phonotactique(mot: str) -> List[ErreurPhono]:
    if mot.lower() in DICTIONNAIRE:
        return []
    erreurs = []
    for regle in REGLES_PHONOTACTIQUES:
        if regle["pattern"].search(mot.strip()):
            erreurs.append(ErreurPhono(mot, regle["id"], regle["description"]))
    return erreurs


# ── Module 3 : Lemmatisation ───────────────────────────────────────────
def appliquer_mutations(mot: str) -> list:
    candidats = []
    for mutation, origines in MUTATIONS.items():
        if mot.startswith(mutation):
            reste = mot[len(mutation):]
            for origine in origines:
                candidat = origine + reste
                if candidat in DICTIONNAIRE:
                    candidats.append(candidat)
    return candidats

def suffixe_na_est_valide(mot: str, racine_candidate: str) -> bool:
    return racine_candidate in DICTIONNAIRE

def lemmatiser(mot: str) -> dict:
    mot_c = mot.lower().strip()

    # N0 : lookup direct MOT_VERS_RACINE
    if mot_c in MOT_VERS_RACINE:
        return {"racine": MOT_VERS_RACINE[mot_c],
                "methode": "lookup_direct", "confiance": "haute"}

    # N0bis : mot est lui-même une racine
    if mot_c in RACINES_TENY:
        return {"racine": mot_c,
                "methode": "racine_directe", "confiance": "haute"}

    # Restauration voyelle finale
    restaure = restaurer_voyelle_finale(mot_c)
    if restaure != mot_c and restaure in MOT_VERS_RACINE:
        return {"racine": MOT_VERS_RACINE[restaure],
                "methode": "lookup_restaure", "confiance": "haute"}

    formes    = [mot_c, restaure] if restaure != mot_c else [mot_c]
    candidats = []

    for forme in formes:
        for pfx in sorted(PREFIXES, key=len, reverse=True):
            if forme.startswith(pfx) and len(forme) > len(pfx) + 2:
                reste = forme[len(pfx):]

                for sfx in sorted(SUFFIXES, key=len, reverse=True):
                    if reste.endswith(sfx) and len(reste) > len(sfx) + 1:
                        racine = reste[:-len(sfx)]
                        if sfx == "na" and not suffixe_na_est_valide(forme, racine):
                            continue
                        if racine in DICTIONNAIRE:
                            candidats.append((racine, len(pfx)+len(sfx), "regles"))

                if reste in DICTIONNAIRE:
                    candidats.append((reste, len(pfx), "regles"))

                for rm in appliquer_mutations(reste):
                    candidats.append((rm, len(pfx)+1, "mutation+prefixe"))

                for sfx in sorted(SUFFIXES, key=len, reverse=True):
                    if reste.endswith(sfx) and len(reste) > len(sfx) + 1:
                        rsf = reste[:-len(sfx)]
                        if sfx == "na" and not suffixe_na_est_valide(forme, rsf):
                            continue
                        for rm in appliquer_mutations(rsf):
                            candidats.append((rm, len(pfx)+len(sfx)+1,
                                              "mutation+prefixe+suffixe"))

        for sfx in sorted(SUFFIXES, key=len, reverse=True):
            if forme.endswith(sfx) and len(forme) > len(sfx) + 2:
                racine = forme[:-len(sfx)]
                if sfx == "na" and not suffixe_na_est_valide(forme, racine):
                    continue
                if racine in DICTIONNAIRE:
                    candidats.append((racine, len(sfx), "regles"))

        for rm in appliquer_mutations(forme):
            candidats.append((rm, 1, "mutation_directe"))

    if candidats:
        meilleur = min(candidats, key=lambda x: len(x[0]))
        return {"racine": meilleur[0], "methode": meilleur[2],
                "confiance": "haute" if meilleur[0] in DICTIONNAIRE else "moyenne"}

    if restaure != mot_c and restaure in DICTIONNAIRE:
        return {"racine": restaure, "methode": "restauration", "confiance": "haute"}

    if mot_c in DICTIONNAIRE:
        return {"racine": mot_c, "methode": "direct", "confiance": "haute"}

    return {"racine": mot_c, "methode": "non_trouvé", "confiance": "faible"}


# ── Pipeline complet ───────────────────────────────────────────────────
def analyser_phrase(texte: str) -> None:
    print("=" * 65)
    print(f"TEXTE : {texte}")
    print("=" * 65)

    tokens = tokeniser(texte)
    print(f"Tokens : {tokens}\n")

    vus = set()
    for mot in tokens:
        if mot in vus:
            continue
        vus.add(mot)

        if mot in STOPWORDS_MG:
            print(f"    '{mot}' — mot grammatical")
            continue

        print(f"\n   '{mot}'")

        # M1
        restaure = restaurer_voyelle_finale(mot.lower())
        if est_correct(mot):
            if restaure != mot.lower() and restaure in DICTIONNAIRE:
                print(f"      Correct (forme complète : '{restaure}')")
            else:
                print(f"      Correct")
        else:
            sugg = suggerer(mot)
            print(f"      Inconnu | Suggestions : {[s for s,_ in sugg] or ['—']}")

        # M2
        for e in verifier_phonotactique(mot):
            print(f"       [{e.regle}] {e.description}")

        # M3
        r = lemmatiser(mot)
        if r["methode"] != "non_trouvé":
            if r["racine"] != mot.lower() and r["racine"] != restaure.lower():
                print(f"      Lemme : '{r['racine']}' "
                      f"({r['methode']}, confiance {r['confiance']})")
            else:
                print(f"      Lemme : '{r['racine']}' (racine de base)")
    print()


# ── Tests ──────────────────────────────────────────────────────────────
analyser_phrase("Ny fitiavana sy ny fampianaran-teny no zava-dehibe")
analyser_phrase("Ny toetr'andro androany dia tsara")
analyser_phrase("Ny fomban-drazana malagasy dia manan-danja")

TEXTE : Ny fitiavana sy ny fampianaran-teny no zava-dehibe
Tokens : ['ny', 'fitiavana', 'sy', 'ny', 'fampianaran', 'teny', 'no', 'zava', 'dehibe']

    'ny' — mot grammatical

   'fitiavana'
      Correct
      Lemme : 'tia' (lookup_direct, confiance haute)
    'sy' — mot grammatical

   'fampianaran'
      Correct (forme complète : 'fampianarana')
      Lemme : 'anatra' (lookup_restaure, confiance haute)

   'teny'
      Correct
      Lemme : 'teny' (racine de base)
    'no' — mot grammatical

   'zava'
      Correct
      Lemme : 'zava' (racine de base)

   'dehibe'
      Correct
      Lemme : 'lehibe' (mutation_directe, confiance haute)

TEXTE : Ny toetr'andro androany dia tsara
Tokens : ['ny', 'toetr', 'andro', 'androany', 'dia', 'tsara']

    'ny' — mot grammatical

   'toetr'
      Correct (forme complète : 'toetra')
       [FIN_CONSONNE] Fin de mot atypique (malagasy finit par voyelle ou -ny)
      Lemme : 'toetra' (racine de base)

   'andro'
      Correct
      Lemme : 'andro'